# Delhi Climate Time Series (Local): MLflow Tracking + Train

Dataset files (same folder as this notebook):
- `DailyDelhiClimateTrain.csv`
- `DailyDelhiClimateTest.csv`

Goal: predict `meantemp` using previous-day (lag-1) features.


## Start MLflow Server (Run This Before Training)

Recommended (project-local DB + artifacts):
```bash
mlflow server --host 127.0.0.1 --port 5000 --backend-store-uri sqlite:///mlruns/mlflow.db --default-artifact-root ./mlruns
```

If you see an out-of-date schema error for an existing DB, upgrade it:
```bash
mlflow db upgrade sqlite:///mlruns/mlflow.db
```

UI:
- http://127.0.0.1:5000


In [7]:
# Install requirements (run once)
# %pip install -U pandas scikit-learn "mlflow<3"


In [8]:
import os
from pathlib import Path

import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error


## Configure MLflow


In [9]:
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("delhi-climate-timeseries-local")

print("MLflow tracking URI:", mlflow.get_tracking_uri())


MLflow tracking URI: http://127.0.0.1:5000


## Load Train/Test CSVs


In [10]:
train_path = Path("DailyDelhiClimateTrain.csv")
test_path = Path("DailyDelhiClimateTest.csv")

if not train_path.exists():
    raise FileNotFoundError(str(train_path.resolve()))
if not test_path.exists():
    raise FileNotFoundError(str(test_path.resolve()))

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

train_df.head()


,date,meantemp,humidity,wind_speed,meanpressure
0,2013-01-01,10.000000,84.500000,0.000000,1015.666667
1,2013-01-02,7.400000,92.000000,2.980000,1017.800000
2,2013-01-03,7.166667,87.000000,4.633333,1018.666667
3,2013-01-04,8.666667,71.333333,1.233333,1017.166667
4,2013-01-05,6.000000,86.833333,3.700000,1016.500000


## Feature Engineering (Lag-1)
We predict `meantemp[t]` from previous day values.


In [11]:
def make_lag1_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    for col in ["meantemp", "humidity", "wind_speed", "meanpressure"]:
        df[f"{col}_lag1"] = df[col].shift(1)

    return df.dropna().reset_index(drop=True)

full = pd.concat([train_df, test_df], ignore_index=True)
full_feat = make_lag1_features(full)

test_start = pd.to_datetime(test_df["date"]).min()
train_feat = full_feat[full_feat["date"] < test_start].copy()
test_feat  = full_feat[full_feat["date"] >= test_start].copy()

feature_cols = ["meantemp_lag1", "humidity_lag1", "wind_speed_lag1", "meanpressure_lag1"]
target_col = "meantemp"

X_train = train_feat[feature_cols]
y_train = train_feat[target_col]
X_test  = test_feat[feature_cols]
y_test  = test_feat[target_col]

X_train.shape, X_test.shape


((1460, 4), (115, 4))

## Train + Log With MLflow


In [12]:
alpha = 1.0
model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=alpha)),
])

with mlflow.start_run(run_name="ridge_lag1") as run:
    mlflow.log_param("alpha", alpha)
    mlflow.log_param("features", ",".join(feature_cols))
    mlflow.log_param("target", target_col)

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    rmse = mean_squared_error(y_test, preds) ** 0.5
    mae = mean_absolute_error(y_test, preds)
    mlflow.log_metric("rmse", float(rmse))
    mlflow.log_metric("mae", float(mae))

    input_example = pd.DataFrame([X_test.iloc[0].to_dict()])
    mlflow.sklearn.log_model(model, artifact_path="model", input_example=input_example)

    run_id = run.info.run_id

print("Run ID:", run_id)
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")


2026/10/24 07:07:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run ridge_lag1 at: http://127.0.0.1:5000/#/experiments/1/runs/b3bcfd02e848442e9a924635b03498e4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID: b3bcfd02e848442e9a924635b03498e4
RMSE: 1.8230
MAE:  1.4031


## Quick Prediction
The model expects lag feature columns.


In [14]:
example_row = X_test.iloc[0:1]
pred = model.predict(example_row)[0]
print("Predicted meantemp:", float(pred))
print("Actual meantemp:", float(y_test.iloc[1]))


Predicted meantemp: 15.416831197697705
Actual meantemp: 15.91304347826087
